In [ ]:
# =========================
# BOOT CELL — SIEMPRE CORRER PRIMERO
# =========================

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

# -------------------------
# 1. reconstruir race_data
# -------------------------
race_data = [
    ("2019-11-03", "marathon"),
    ("2019-11-17", "half"),
    ("2020-07-19", "race"),
    ("2020-10-11", "race"),
    ("2021-07-18", "race"),
    ("2021-12-12", "marathon"),
    ("2022-07-24", "race"),
    ("2022-09-04", "half"),
    ("2022-11-06", "marathon"),
    ("2022-12-04", "half"),
    ("2023-02-26", "half"),
    ("2023-03-26", "half"),
    ("2023-09-03", "half"),
    ("2023-11-05", "marathon"),
    ("2023-12-03", "half"),
    ("2024-04-07", "race"),
    ("2024-11-24", "marathon"),
    ("2025-02-23", "half"),
    ("2025-05-03", "10k"),
    ("2025-07-27", "10k"),
    ("2025-09-07", "half"),
    ("2025-11-30", "marathon"),

    # 👇 quitamos HM 2026
    # ("2026-02-22", "half"),

    # 👇 agregamos 3000m
    ("2024-12-06", "3k")
]

df_races = pd.DataFrame(race_data, columns=["date", "race_type"])
df_races["date"] = pd.to_datetime(df_races["date"])

print("BOOT OK → df_races:", df_races.shape)

In [ ]:
race_data = [
    ("2019-11-03", "marathon"),
    ("2019-11-17", "half"),
    ("2020-07-19", "race"),
    ("2020-10-11", "race"),
    ("2021-07-18", "race"),
    ("2021-12-12", "marathon"),
    ("2022-07-24", "race"),
    ("2022-09-04", "half"),
    ("2022-11-06", "marathon"),
    ("2022-12-04", "half"),
    ("2023-02-26", "half"),
    ("2023-03-26", "half"),
    ("2023-09-03", "half"),
    ("2023-11-05", "marathon"),
    ("2023-12-03", "half"),
    ("2024-04-07", "race"),
    ("2024-11-24", "marathon"),
    ("2025-02-23", "half"),
    ("2025-05-03", "10k"),
    ("2025-07-27", "10k"),
    ("2025-09-07", "half"),
    ("2025-11-30", "marathon"),
    ("2026-02-22", "half"),

    # 👇 NUEVO
    ("2024-12-06", "3k")
]

In [ ]:
# ===== BOOT SEQUENCE =====
import numpy as np
import pandas as pd
from pathlib import Path
import json
from collections import Counter

# librerías externas
try:
    from sklearn.linear_model import LinearRegression
except:
    !pip install scikit-learn
    from sklearn.linear_model import LinearRegression

print("✅ Entorno listo")

In [ ]:
# ===== DATA LOAD =====
json_dir = Path(r"C:\Users\Quantuminfo\Desktop\PROYECTO BIOQUANTUM\DATOS JASON POLAR OCT2025-MAR2026")

files = list(json_dir.glob("*.json"))
training_files = [f for f in files if "training-session" in f.name]

print(f"Archivos totales: {len(files)}")
print(f"Training sessions: {len(training_files)}")

In [ ]:
# ===== PARSER =====
def parse_training_session(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ex = data["exercises"][0]

    return {
        "name": data.get("name", "polar_session"),
        "session_date": data.get("startTime"),
        "distance_km": ex.get("distanceMeters", 0) / 1000,
        "duration_s": ex.get("durationMillis", 0) / 1000,
        "avg_hr": data.get("hrAvg"),
        "max_hr": data.get("hrMax"),
        "training_load": ex.get("trainingLoad"),
    }

print("✅ Parser listo")

In [ ]:
# ===== BUILD DATASET =====
sessions = [parse_training_session(f) for f in training_files]
df_all = pd.DataFrame(sessions)

df_all["session_date"] = pd.to_datetime(df_all["session_date"], errors="coerce")
df_all["avg_hr"] = pd.to_numeric(df_all["avg_hr"], errors="coerce")
df_all["max_hr"] = pd.to_numeric(df_all["max_hr"], errors="coerce")

df_all["duration_min"] = df_all["duration_s"] / 60
df_all["pace_min_km"] = df_all["duration_min"] / df_all["distance_km"]
df_all["days_ago"] = (df_all["session_date"].max() - df_all["session_date"]).dt.days

df_model = df_all[
    (df_all["distance_km"] >= 3) &
    (df_all["distance_km"] <= 43) &
    (df_all["pace_min_km"].between(3, 8)) &
    (df_all["avg_hr"].between(120, 190))
].copy()

print(df_model.shape)

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import numpy as np
import pandas as pd

def minmax_clip(x, lo=0.0, hi=1.0):
    return np.clip(x, lo, hi)

def compute_days_ago(df, date_col="session_date"):
    today = pd.Timestamp.today().normalize()
    df["session_date"] = pd.to_datetime(df[date_col])
    df["days_ago"] = (today - df["session_date"]).dt.days
    return df

def compute_week_start(df, date_col="session_date"):
    df["week_start"] = pd.to_datetime(df[date_col]) - pd.to_timedelta(pd.to_datetime(df[date_col]).dt.weekday, unit="D")
    return df

def compute_validity_flags(df):
    df["has_distance"] = df["distance_km"].notna() & (df["distance_km"] > 0)
    df["has_duration"] = df["duration_min"].notna() & (df["duration_min"] > 0)

    df["valid_pace_hr"] = (
        df["has_distance"]
        & df["has_duration"]
        & df["avg_hr"].notna()
        & (df["avg_hr"] >= 90)
        & (df["avg_hr"] <= 190)
        & df["pace_min_km"].notna()
        & (df["pace_min_km"] >= 2.5)
        & (df["pace_min_km"] <= 9.0)
    ).astype(int)

    df["is_complete_session"] = (
        df["has_distance"] & df["has_duration"] & (df["valid_pace_hr"] == 1)
    ).astype(int)

    return df

def compute_recency_score(df):
    conditions = [
        df["days_ago"] <= 21,
        (df["days_ago"] > 21) & (df["days_ago"] <= 42),
        (df["days_ago"] > 42) & (df["days_ago"] <= 84),
        (df["days_ago"] > 84) & (df["days_ago"] <= 140),
        (df["days_ago"] > 140),
    ]
    values = [1.00, 0.85, 0.65, 0.45, 0.25]
    df["recency_score"] = np.select(conditions, values, default=0.25)
    return df

def compute_relevance_flag(df):
    # irrelevantes
    irrelevant_mask = (
        (df["valid_pace_hr"] == 0)
        | (df["distance_km"] < 6)
        | (df["duration_min"] < 35)
    )

    # candidatas
    candidate_mask = (
        (df["valid_pace_hr"] == 1)
        & (df["distance_km"] >= 8)
        & (df["duration_min"] >= 45)
        & (df["distance_km"] <= 35)
    )

    df["relevance_flag"] = "irrelevant"
    df.loc[candidate_mask, "relevance_flag"] = "candidate"

    # high signal provisional
    high_signal_mask = (
        candidate_mask
        & (df["distance_km"].between(10, 30))
        & (df["steady_effort_score"].fillna(0.5) >= 0.60)
        & (df["contamination_score"].fillna(0.0) <= 0.35)
        & (df["week_weight"].fillna(1.0) >= 0.60)
    )

    df.loc[high_signal_mask, "relevance_flag"] = "high_signal"

    score_map = {
        "irrelevant": 0.0,
        "candidate": 0.6,
        "high_signal": 1.0
    }
    df["relevance_score"] = df["relevance_flag"].map(score_map).astype(float)
    return df

def compute_distance_specificity(df):
    # 10K
    df["specificity_10k"] = 0.0
    df.loc[df["distance_km"].between(8, 14), "specificity_10k"] = 1.0
    df.loc[df["distance_km"].between(6, 16), "specificity_10k"] = np.maximum(
        df.loc[df["distance_km"].between(6, 16), "specificity_10k"], 0.75
    )

    # HM
    df["specificity_hm"] = 0.0
    df.loc[df["distance_km"].between(12, 24), "specificity_hm"] = 1.0
    df.loc[df["distance_km"].between(10, 28), "specificity_hm"] = np.maximum(
        df.loc[df["distance_km"].between(10, 28), "specificity_hm"], 0.75
    )

    # Marathon
    df["specificity_marathon"] = 0.0
    df.loc[df["distance_km"].between(24, 32), "specificity_marathon"] = 1.0
    df.loc[df["distance_km"].between(20, 35), "specificity_marathon"] = np.maximum(
        df.loc[df["distance_km"].between(20, 35), "specificity_marathon"], 0.75
    )

    return df

def compute_initial_scores(df):
    # Si aún no tienes estas columnas, las creamos provisionales
    if "steady_effort_score" not in df.columns:
        df["steady_effort_score"] = 0.70

    if "efficiency_score" not in df.columns:
        df["efficiency_score"] = 0.70

    if "contamination_score" not in df.columns:
        df["contamination_score"] = 0.0

    if "week_weight" not in df.columns:
        df["week_weight"] = 1.0

    return df

def compute_informativeness(df):
    df["informativeness_10k"] = (
        df["relevance_score"]
        * df["steady_effort_score"]
        * df["efficiency_score"]
        * df["recency_score"]
        * df["specificity_10k"]
        * (1 - df["contamination_score"])
        * df["week_weight"]
    )

    df["informativeness_hm"] = (
        df["relevance_score"]
        * df["steady_effort_score"]
        * df["efficiency_score"]
        * df["recency_score"]
        * df["specificity_hm"]
        * (1 - df["contamination_score"])
        * df["week_weight"]
    )

    df["informativeness_marathon"] = (
        df["relevance_score"]
        * df["steady_effort_score"]
        * df["efficiency_score"]
        * df["recency_score"]
        * df["specificity_marathon"]
        * (1 - df["contamination_score"])
        * df["week_weight"]
    )

    return df

def build_race_predictor_v2_base(df):
    df = df.copy()

    df = compute_days_ago(df, "session_date")
    df = compute_week_start(df, "session_date")
    df = compute_validity_flags(df)
    df = compute_initial_scores(df)
    df = compute_recency_score(df)
    df = compute_distance_specificity(df)
    df = compute_relevance_flag(df)
    df = compute_informativeness(df)

    return df

In [ ]:
import pandas as pd

In [ ]:
data = [
    {"session_date": "2025-10-01", "distance_km": 10.3, "duration_min": 38.0, "pace_min_km": 3.68, "avg_hr": 151},
    {"session_date": "2025-10-05", "distance_km": 21.0, "duration_min": 95.0, "pace_min_km": 4.52, "avg_hr": 148},
    {"session_date": "2025-10-12", "distance_km": 28.0, "duration_min": 130.0, "pace_min_km": 4.64, "avg_hr": 145},
]

df_sessions = pd.DataFrame(data)

df_sessions

In [ ]:
df_v2 = build_race_predictor_v2_base(df_sessions)

df_v2

In [ ]:
from src.analysis.session_dataframe import sessions_to_dataframe

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
from src.analysis.session_dataframe import sessions_to_dataframe

In [ ]:
sessions_to_dataframe

In [ ]:
from src.ingestion.polar_mock import session_from_polar_dict

In [ ]:
s = session_from_polar_dict({
    "distance_km": 10.3,
    "duration_s": 2381,
    "avg_hr": 151,
    "name": "Test 10K"
})

In [ ]:
s

In [ ]:
df_sessions = sessions_to_dataframe([s])
df_sessions

In [ ]:
df_v2 = build_race_predictor_v2_base(df_sessions)
df_v2

In [ ]:
s = session_from_polar_dict({
    "distance_km": 10.3,
    "duration_s": 2381,
    "avg_hr": 151,
    "name": "Test 10K"
})

df_sessions = sessions_to_dataframe([s])

df_sessions

In [ ]:
from src.ingestion.polar_mock import session_from_polar_dict

In [ ]:
from src.ingestion.polar_mock import session_from_polar_dict

In [ ]:
s = session_from_polar_dict({
    "distance_km": 10.3,
    "duration_s": 2381,
    "avg_hr": 151,
    "name": "Test 10K"
})

In [ ]:
df_sessions = sessions_to_dataframe([s])
df_sessions

In [ ]:
from src.analysis.session_dataframe import sessions_to_dataframe

In [ ]:
df_v2 = build_race_predictor_v2_base(df_sessions)
df_v2

In [ ]:
df_sessions["session_date"] = "2025-10-01"
df_sessions

In [ ]:
df_v2 = build_race_predictor_v2_base(df_sessions)
df_v2

In [ ]:
df_v2.columns

In [ ]:
df_v2 = build_race_predictor_v2_base(df_sessions)
df_v2

In [ ]:
df_v2[[
    "distance_km",
    "pace_min_km",
    "avg_hr",
    "relevance_flag",
    "informativeness_10k",
    "informativeness_hm",
    "informativeness_marathon"
]]

In [ ]:
df_v2["relevance_flag"].value_counts()

In [ ]:
df_v2[[
    "distance_km",
    "relevance_flag",
    "relevance_score",
    "specificity_10k",
    "specificity_hm",
    "specificity_marathon"
]]

In [ ]:
from pathlib import Path

json_dir = Path(r"C:\Users\Quantuminfo\Desktop\PROYECTO BIOQUANTUM\DATOS JASON POLAR OCT2025-MAR2026")
list(json_dir.glob("*.json"))[:5]

In [ ]:
from collections import Counter

files = list(json_dir.glob("*.json"))
prefixes = [f.name.split("_")[0] for f in files]

Counter(prefixes)

In [ ]:
from collections import Counter

files = list(json_dir.glob("*.json"))

def file_family(name: str) -> str:
    if name.startswith("training-session"):
        return "training-session"
    if name.startswith("247ohr"):
        return "247ohr"
    if name.startswith("ppi"):
        return "ppi"
    if name.startswith("nightly"):
        return "nightly"
    if name.startswith("sleep"):
        return "sleep"
    if name.startswith("activity"):
        return "activity"
    if name.startswith("account-data"):
        return "account-data"
    if name.startswith("account-profile"):
        return "account-profile"
    return "other"

family_counts = Counter(file_family(f.name) for f in files)
family_counts

In [ ]:
other_files = [f.name for f in files if file_family(f.name) == "other"]
other_files[:50]

In [ ]:
training_files = [f for f in files if f.name.startswith("training-session")]
training_files[:3]

In [ ]:
import json

with open(training_files[0], "r", encoding="utf-8") as f:
    session_json = json.load(f)

session_json

In [ ]:
session_json.keys()

In [ ]:
session_json["sport"], type(session_json["exercises"]), len(session_json["exercises"])

In [ ]:
session_json["exercises"][0].keys()

In [ ]:
def parse_training_session(file_path):
    import json
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ex = data["exercises"][0]

    duration_s = ex.get("durationMillis", 0) / 1000
    distance_km = ex.get("distanceMeters", 0) / 1000

    hr_avg = data.get("hrAvg")
    hr_max = data.get("hrMax")

    return {
        "name": data.get("name", "polar_session"),
        "session_date": data.get("startTime"),
        "distance_km": distance_km,
        "duration_s": duration_s,
        "avg_hr": hr_avg,
        "max_hr": hr_max,
        "training_load": ex.get("trainingLoad"),
    }

In [ ]:
parse_training_session(training_files[0])

In [ ]:
sessions = [parse_training_session(f) for f in training_files]
len(sessions)

In [ ]:
import pandas as pd

df_all = pd.DataFrame(sessions)

df_all.head()

In [ ]:
df_all["session_date"] = pd.to_datetime(df_all["session_date"])
df_all.dtypes

In [ ]:
df_all["duration_min"] = df_all["duration_s"] / 60
df_all["pace_min_km"] = df_all["duration_min"] / df_all["distance_km"]

df_all.head()

In [ ]:
today = df_all["session_date"].max()

df_all["days_ago"] = (today - df_all["session_date"]).dt.days

df_all.head()

In [ ]:
df_all["week_start"] = df_all["session_date"].dt.to_period("W").apply(lambda r: r.start_time)

df_all.head()

In [ ]:
df_all["distance_km"].describe()

In [ ]:
df_run = df_all[df_all["distance_km"] >= 3]

len(df_run)

In [ ]:
weekly_km = df_run.groupby("week_start")["distance_km"].sum()

weekly_km.describe()

In [ ]:
df_run["pace_min_km"].describe()

In [ ]:
df_run = df_run[df_run["pace_min_km"] < 8]
len(df_run)

In [ ]:
df_run = df_run[df_run["distance_km"] <= 45]

df_run["distance_km"].max()

In [ ]:
df_run["distance_km"].describe()

In [ ]:
df_run[df_run["distance_km"] > 25][["session_date","distance_km","pace_min_km"]].sort_values("distance_km", ascending=False).head(10)

In [ ]:
df_run["year"] = df_run["session_date"].dt.year
df_run["year"].value_counts().sort_index()

In [ ]:
df_run.groupby("year")["distance_km"].sum().sort_index()

In [ ]:
weekly_by_year = df_run.groupby(["year", "week_start"])["distance_km"].sum()

weekly_by_year.groupby("year").max()

In [ ]:
df_run.groupby("year")["pace_min_km"].median()

In [ ]:
long_runs = df_run[df_run["distance_km"] >= 18]

long_runs.groupby("year")["pace_min_km"].median()

In [ ]:
weekly_km = df_run.groupby(["year","week_start"])["distance_km"].sum()

weekly_km.groupby("year").mean()

In [ ]:
(df_run["pace_min_km"] * df_run["distance_km"]).sum() / df_run["distance_km"].sum()

In [ ]:
df_hr = df_run[df_run["avg_hr"].notna()]

(df_hr["pace_min_km"] * df_hr["distance_km"]).sum() / df_hr["distance_km"].sum()

In [ ]:
df_hr["avg_hr"].mean()

In [ ]:
df_all[["avg_hr","max_hr"]].notna().sum()

In [ ]:
session_json["exercises"][0]["hrAvg"], session_json["exercises"][0]["hrMax"]

In [ ]:
session_json["hrAvg"], session_json["hrMax"]

In [ ]:
def parse_training_session(file_path):
    import json
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ex = data["exercises"][0]

    duration_s = ex.get("durationMillis", 0) / 1000
    distance_km = ex.get("distanceMeters", 0) / 1000

    hr_avg = data.get("hrAvg")
    hr_max = data.get("hrMax")

    return {
        "name": data.get("name", "polar_session"),
        "session_date": data.get("startTime"),
        "distance_km": distance_km,
        "duration_s": duration_s,
        "avg_hr": hr_avg,
        "max_hr": hr_max,
        "training_load": ex.get("trainingLoad"),
    }

In [ ]:
sessions = [parse_training_session(f) for f in training_files]
df_all = pd.DataFrame(sessions)

df_all[["avg_hr", "max_hr"]].notna().sum()

In [ ]:
df_run["efficiency"] = df_run["pace_min_km"] / df_run["avg_hr"]

df_run["efficiency"].describe()

In [ ]:
df_all["avg_hr"] = pd.to_numeric(df_all["avg_hr"], errors="coerce")
df_all["max_hr"] = pd.to_numeric(df_all["max_hr"], errors="coerce")

In [ ]:
df_run["efficiency"] = df_run["pace_min_km"] / df_run["avg_hr"]

df_run["efficiency"].describe()

In [ ]:
df_run["avg_hr"] = pd.to_numeric(df_run["avg_hr"], errors="coerce")
df_run["max_hr"] = pd.to_numeric(df_run["max_hr"], errors="coerce")

df_run["efficiency"] = df_run["pace_min_km"] / df_run["avg_hr"]

df_run["efficiency"].describe()

In [ ]:
df_all["avg_hr"] = pd.to_numeric(df_all["avg_hr"], errors="coerce")
df_all["max_hr"] = pd.to_numeric(df_all["max_hr"], errors="coerce")

df_all["duration_min"] = df_all["duration_s"] / 60
df_all["pace_min_km"] = df_all["duration_min"] / df_all["distance_km"]
df_all["days_ago"] = (df_all["session_date"].max() - df_all["session_date"]).dt.days
df_all["week_start"] = df_all["session_date"].dt.to_period("W").apply(lambda r: r.start_time)

df_run = df_all[
    (df_all["distance_km"] >= 3) &
    (df_all["pace_min_km"] < 8) &
    (df_all["distance_km"] <= 42)
].copy()

df_run["efficiency"] = df_run["pace_min_km"] / df_run["avg_hr"]

df_run["efficiency"].describe()

In [ ]:
df_all["session_date"] = pd.to_datetime(df_all["session_date"], errors="coerce")

In [ ]:
df_all["avg_hr"] = pd.to_numeric(df_all["avg_hr"], errors="coerce")
df_all["max_hr"] = pd.to_numeric(df_all["max_hr"], errors="coerce")

df_all["duration_min"] = df_all["duration_s"] / 60
df_all["pace_min_km"] = df_all["duration_min"] / df_all["distance_km"]
df_all["days_ago"] = (df_all["session_date"].max() - df_all["session_date"]).dt.days
df_all["week_start"] = df_all["session_date"].dt.to_period("W").apply(lambda r: r.start_time)

df_run = df_all[
    (df_all["distance_km"] >= 3) &
    (df_all["pace_min_km"] < 8) &
    (df_all["distance_km"] <= 42)
].copy()

df_run["efficiency"] = df_run["pace_min_km"] / df_run["avg_hr"]

df_run["efficiency"].describe()

In [ ]:
df_run["efficiency"] = df_run["pace_min_km"] / df_run["avg_hr"]

In [ ]:
df_run.groupby(df_run["session_date"].dt.year)["efficiency"].median()

In [ ]:
long_runs = df_run[df_run["distance_km"] >= 18]

long_runs.groupby(long_runs["session_date"].dt.year)["efficiency"].median()

In [ ]:
long_runs.groupby(long_runs["session_date"].dt.year).size()

In [ ]:
long_runs.sort_values("efficiency").head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr","efficiency"]
]

In [ ]:
long_runs = df_run[df_run["distance_km"] >= 18].copy()

long_runs.sort_values("efficiency").head(10)[
    ["session_date", "distance_km", "pace_min_km", "avg_hr", "efficiency"]
]

In [ ]:
df_run["efficiency"] = df_run["pace_min_km"] / df_run["avg_hr"]

long_runs = df_run[df_run["distance_km"] >= 18].copy()

long_runs.sort_values("efficiency").head(10)[
    ["session_date", "distance_km", "pace_min_km", "avg_hr", "efficiency"]
]

In [ ]:
medium_runs = df_run[(df_run["distance_km"] >= 10) & (df_run["distance_km"] <= 16)].copy()

medium_runs.sort_values("efficiency").head(10)[
    ["session_date", "distance_km", "pace_min_km", "avg_hr", "efficiency"]
]

In [ ]:
df_run.sort_values("pace_min_km").head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr"]
]

In [ ]:
df_run.groupby(pd.cut(df_run["distance_km"], [3,6,10,15,21,30,42]))["pace_min_km"].median()

In [ ]:
df_run.groupby(pd.cut(df_run["distance_km"], [3,6,10,15,21,30,42]))["avg_hr"].median()

In [ ]:
long_runs[["distance_km","pace_min_km","avg_hr"]].sort_values("distance_km").tail(20)

In [ ]:
hm = df_run[(df_run["distance_km"] >= 20) & (df_run["distance_km"] <= 22)]

hm.sort_values("pace_min_km").head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr"]
]

In [ ]:
marathon = df_run[(df_run["distance_km"] >= 41) & (df_run["distance_km"] <= 43)]
marathon.sort_values("pace_min_km").head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr"]
]

In [ ]:
df_run["performance_score"] = (
    (1 / df_run["pace_min_km"]) *
    (df_run["distance_km"] ** 0.3) *
    (df_run["avg_hr"] / 150)
)

df_run.sort_values("performance_score", ascending=False).head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr","performance_score"]
]

In [ ]:
recent = df_run[df_run["days_ago"] < 120]

recent.sort_values("performance_score", ascending=False).head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr","performance_score"]
]

In [ ]:
clean_recent = recent[
    (recent["avg_hr"] < 165) &
    (recent["pace_min_km"] < 4.8)
]

clean_recent.sort_values("performance_score", ascending=False).head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr","performance_score"]
]

In [ ]:
# 1️⃣ Crear performance_score
df_run["performance_score"] = (
    (1 / df_run["pace_min_km"]) *
    (df_run["distance_km"] ** 0.3) *
    (df_run["avg_hr"] / 150)
)

# 2️⃣ Crear recent
recent = df_run[df_run["days_ago"] < 120]

# 3️⃣ Filtrar limpio
clean_recent = recent[
    (recent["avg_hr"] < 165) &
    (recent["pace_min_km"] < 4.8)
]

# 4️⃣ Mostrar resultados
clean_recent.sort_values("performance_score", ascending=False).head(10)[
    ["session_date","distance_km","pace_min_km","avg_hr","performance_score"]
]

In [ ]:
def build_recent(df):
    df["performance_score"] = ...
    recent = df[df["days_ago"] < 120]
    return recent

In [ ]:
df_run[
    (df_run["avg_hr"].between(145,155)) &
    (df_run["distance_km"].between(6,15))
]["pace_min_km"].median()

In [ ]:
df_model = df_all[
    (df_all["distance_km"] >= 3) &
    (df_all["distance_km"] <= 42) &
    (df_all["pace_min_km"].between(3, 8))
].copy()

# eliminar registros sin FC
df_model = df_model[df_model["avg_hr"].notna()]

# eliminar valores absurdos
df_model = df_model[
    (df_model["avg_hr"] > 120) &
    (df_model["avg_hr"] < 190)
]

df_model.shape

In [ ]:
df_model["speed_kmh"] = 60 / df_model["pace_min_km"]
df_model["efficiency"] = df_model["speed_kmh"] / df_model["avg_hr"]
df_model["year"] = df_model["session_date"].dt.year
df_model["week"] = df_model["session_date"].dt.isocalendar().week

In [ ]:
df_model.head()

In [ ]:
df_model.describe()

In [ ]:
df_model.info()

In [ ]:
import numpy as np

# Modelo base: relación logarítmica distancia vs ritmo
df_model["log_distance"] = np.log(df_model["distance_km"])

# Ver correlación
df_model[["pace_min_km", "log_distance"]].corr()

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_model[["log_distance"]]
y = df_model["pace_min_km"]

model = LinearRegression()
model.fit(X, y)

model.coef_, model.intercept_

In [ ]:
def predict_pace(km):
    return model.predict([[np.log(km)]])[0]

predict_pace(21.1), predict_pace(42.2)

In [ ]:
print("Jupyter sigue vivo")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json

In [ ]:
from collections import Counter
from sklearn.linear_model import LinearRegression

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
from collections import Counter
from sklearn.linear_model import LinearRegression

In [ ]:
df_model["log_distance"] = np.log(df_model["distance_km"])

df_model[["pace_min_km", "log_distance"]].corr()

In [ ]:
import json

def parse_training_session(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ex = data["exercises"][0]

    duration_s = ex.get("durationMillis", 0) / 1000
    distance_km = ex.get("distanceMeters", 0) / 1000

    hr_avg = data.get("hrAvg")
    hr_max = data.get("hrMax")

    return {
        "name": data.get("name", "polar_session"),
        "session_date": data.get("startTime"),
        "distance_km": distance_km,
        "duration_s": duration_s,
        "avg_hr": hr_avg,
        "max_hr": hr_max,
        "training_load": ex.get("trainingLoad"),
    }

In [ ]:
from pathlib import Path

json_dir = Path(r"C:\Users\Quantuminfo\Desktop\PROYECTO BIOQUANTUM\DATOS JASON POLAR OCT2025-MAR2026")

files = list(json_dir.glob("*.json"))

len(files)

In [ ]:
training_files = [f for f in files if "training-session" in f.name]

len(training_files)

In [ ]:
import json

def parse_training_session(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ex = data["exercises"][0]

    duration_s = ex.get("durationMillis", 0) / 1000
    distance_km = ex.get("distanceMeters", 0) / 1000

    hr_avg = data.get("hrAvg")
    hr_max = data.get("hrMax")

    return {
        "name": data.get("name", "polar_session"),
        "session_date": data.get("startTime"),
        "distance_km": distance_km,
        "duration_s": duration_s,
        "avg_hr": hr_avg,
        "max_hr": hr_max,
        "training_load": ex.get("trainingLoad"),
    }

In [ ]:
sessions = [parse_training_session(f) for f in training_files]
len(sessions)

In [ ]:
df_all = pd.DataFrame(sessions)

df_all["session_date"] = pd.to_datetime(df_all["session_date"], errors="coerce")
df_all["avg_hr"] = pd.to_numeric(df_all["avg_hr"], errors="coerce")
df_all["max_hr"] = pd.to_numeric(df_all["max_hr"], errors="coerce")

df_all["duration_min"] = df_all["duration_s"] / 60
df_all["pace_min_km"] = df_all["duration_min"] / df_all["distance_km"]
df_all["days_ago"] = (df_all["session_date"].max() - df_all["session_date"]).dt.days
df_all["week_start"] = df_all["session_date"].dt.to_period("W").apply(lambda r: r.start_time)

df_model = df_all[
    (df_all["distance_km"] >= 3) &
    (df_all["distance_km"] <= 42) &
    (df_all["pace_min_km"].between(3, 8)) &
    (df_all["avg_hr"].between(120, 190))
].copy()

df_model["speed_kmh"] = 60 / df_model["pace_min_km"]
df_model["efficiency"] = df_model["speed_kmh"] / df_model["avg_hr"]
df_model["year"] = df_model["session_date"].dt.year
df_model["week"] = df_model["session_date"].dt.isocalendar().week

df_model.shape

In [ ]:
df_model["log_distance"] = np.log(df_model["distance_km"])

df_model[["pace_min_km", "log_distance"]].corr()

In [ ]:
df_perf = df_model[
    (df_model["distance_km"] >= 8) &
    (df_model["pace_min_km"] < 4.5) &
    (df_model["avg_hr"] > 150)
].copy()

df_perf.shape


In [ ]:
df_perf["log_distance"] = np.log(df_perf["distance_km"])

df_perf[["pace_min_km", "log_distance"]].corr()

In [ ]:
race_data = [
    ("2019-11-03", "marathon"),
    ("2019-11-17", "half"),
    
    ("2020-07-19", "race"),
    ("2020-10-11", "race"),
    
    ("2021-07-18", "race"),
    ("2021-12-12", "marathon"),
    
    ("2022-07-24", "race"),
    ("2022-09-04", "half"),
    ("2022-11-06", "marathon"),
    ("2022-12-04", "half"),
    
    ("2023-02-26", "half"),
    ("2023-03-26", "half"),
    ("2023-09-03", "half"),
    ("2023-11-05", "marathon"),
    ("2023-12-03", "half"),
    
    ("2024-04-07", "race"),
    ("2024-11-24", "marathon"),
    
    ("2025-02-23", "half"),
    ("2025-05-03", "10k"),
    ("2025-07-27", "10k"),
    ("2025-09-07", "half"),
    ("2025-11-30", "marathon"),
    
    ("2026-02-22", "half"),
]

df_races = pd.DataFrame(race_data, columns=["date", "race_type"])
df_races["date"] = pd.to_datetime(df_races["date"])

In [ ]:
print("df_model existe:", "df_model" in globals())
print("shape:", df_model.shape if "df_model" in globals() else None)

In [ ]:
race_data = [
    ("2019-11-03", "marathon"),
    ("2019-11-17", "half"),
    ("2020-07-19", "race"),
    ("2020-10-11", "race"),
    ("2021-07-18", "race"),
    ("2021-12-12", "marathon"),
    ("2022-07-24", "race"),
    ("2022-09-04", "half"),
    ("2022-11-06", "marathon"),
    ("2022-12-04", "half"),
    ("2023-02-26", "half"),
    ("2023-03-26", "half"),
    ("2023-09-03", "half"),
    ("2023-11-05", "marathon"),
    ("2023-12-03", "half"),
    ("2024-04-07", "race"),
    ("2024-11-24", "marathon"),
    ("2025-02-23", "half"),
    ("2025-05-03", "10k"),
    ("2025-07-27", "10k"),
    ("2025-09-07", "half"),
    ("2025-11-30", "marathon"),
    ("2026-02-22", "half"),
]

df_races = pd.DataFrame(race_data, columns=["date", "race_type"])
df_races["date"] = pd.to_datetime(df_races["date"])

df_races

In [ ]:
df_model["race_flag"] = df_model["session_date"].dt.normalize().isin(df_races["date"])

df_model["race_flag"].sum()

In [ ]:
df_model[df_model["race_flag"] == True][[
    "session_date",
    "distance_km",
    "pace_min_km",
    "avg_hr"
]].sort_values("session_date")

In [ ]:
# crear set de fechas objetivo (±1 día)
race_dates_expanded = set()

for d in df_races["date"]:
    race_dates_expanded.add(d)
    race_dates_expanded.add(d + pd.Timedelta(days=1))
    race_dates_expanded.add(d - pd.Timedelta(days=1))

# marcar carreras
df_model["race_flag"] = df_model["session_date"].dt.normalize().isin(race_dates_expanded)

df_model["race_flag"].sum()


In [ ]:
df_model["date_only"] = df_model["session_date"].dt.normalize()

df_model["race_flag"] = df_model["date_only"].isin(df_races["date"])

df_model["race_flag"].sum()

In [ ]:
# fechas de carreras que NO encontraron match
missing = df_races[~df_races["date"].isin(df_model["date_only"])]

missing

In [ ]:
df_model[df_model["distance_km"] > 40][[
    "session_date","distance_km","pace_min_km"
]].sort_values("session_date")

In [ ]:
df_model[df_model["distance_km"] > 40][[
    "session_date","distance_km","pace_min_km"
]].sort_values("session_date")

In [ ]:
df_model[df_model["distance_km"] > 40][[
    "session_date","distance_km","pace_min_km"
]].sort_values("session_date")

In [ ]:
df_model["race_flag"] = df_model["session_date"].dt.normalize().isin(df_races["date"])

df_model["race_flag"].sum()

In [ ]:
df_model["race_flag"] = df_model["session_date"].dt.normalize().isin(df_races["date"])

df_model["race_flag"].sum()

In [ ]:
df_race_only = df_model[df_model["race_flag"]].copy()

df_race_only.sort_values("distance_km")

In [ ]:
df_race_only[[
    "session_date",
    "distance_km",
    "pace_min_km",
    "avg_hr"
]].sort_values("session_date")

In [ ]:
df_race_only["log_distance"] = np.log(df_race_only["distance_km"])

df_race_only[["pace_min_km", "log_distance"]].corr()

In [ ]:
df_race_only["efficiency"] = (60 / df_race_only["pace_min_km"]) / df_race_only["avg_hr"]

df_race_only[["pace_min_km","efficiency"]].corr()

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_race_only[["efficiency"]]
y = df_race_only["pace_min_km"]

model_eff = LinearRegression()
model_eff.fit(X, y)

model_eff.coef_, model_eff.intercept_

In [ ]:
def predict_pace_from_eff(eff):
    return model_eff.predict([[eff]])[0]

predict_pace_from_eff(0.032)

In [ ]:
df_race_only["efficiency"].describe()

In [ ]:
predict_pace_from_eff(0.024)
predict_pace_from_eff(0.025)
predict_pace_from_eff(0.026)

In [ ]:
def predict_pace_from_eff(eff):
    return model_eff.predict(pd.DataFrame({"efficiency": [eff]}))[0]

In [ ]:
predict_pace_from_eff(0.024)
predict_pace_from_eff(0.025)
predict_pace_from_eff(0.026)

In [ ]:
print(predict_pace_from_eff(0.024))
print(predict_pace_from_eff(0.025))
print(predict_pace_from_eff(0.026))

In [ ]:
for e in [0.024, 0.025, 0.026]:
    print(e, "→", predict_pace_from_eff(e))

In [ ]:
df_race_only[["efficiency","pace_min_km"]].head(10)

In [ ]:
df_race_only["efficiency"].describe()

In [ ]:
print(predict_pace_from_eff(0.085))
print(predict_pace_from_eff(0.090))
print(predict_pace_from_eff(0.095))

In [ ]:
df_race_only[["efficiency", "pace_min_km"]].sort_values("efficiency")

In [ ]:
race_data.append(("2024-12-06", "3k"))  # ← pon la fecha real

df_races = pd.DataFrame(race_data, columns=["date", "race_type"])
df_races["date"] = pd.to_datetime(df_races["date"])

In [ ]:
df_races = pd.DataFrame(race_data, columns=["date", "race_type"])
df_races["date"] = pd.to_datetime(df_races["date"])

In [ ]:
df_races.tail()

In [ ]:
df_races

In [ ]:
df_race_only.head()

In [ ]:
print(model_eff)

In [ ]:
def classify_race(distance):
    if distance <= 5:
        return "speed"
    elif distance <= 15:
        return "threshold"
    else:
        return "endurance"

df_race_only["race_profile"] = df_race_only["distance_km"].apply(classify_race)

In [ ]:
def classify_race(distance):
    if distance <= 5:
        return "speed"
    elif distance <= 15:
        return "threshold"
    else:
        return "endurance"

df_race_only["race_profile"] = df_race_only["distance_km"].apply(classify_race)

In [ ]:
df_race_only.groupby("race_profile")["distance_km"].count()

In [ ]:
df_race_only.groupby("race_profile").agg({
    "efficiency": ["mean", "std"],
    "pace_min_km": ["mean", "std"]
})

In [ ]:
df_race_only["pace_pred"] = model_eff.predict(df_race_only[["efficiency"]])

df_race_only["error_seg_km"] = (df_race_only["pace_pred"] - df_race_only["pace_min_km"]) * 60
df_race_only["abs_error_seg_km"] = df_race_only["error_seg_km"].abs()

df_race_only.groupby("race_profile").agg({
    "abs_error_seg_km": "mean",
    "error_seg_km": "mean"
})

In [ ]:
bias_by_profile = df_race_only.groupby("race_profile")["error_seg_km"].mean()
bias_by_profile

In [ ]:
bias_by_profile = df_race_only.groupby("race_profile")["error_seg_km"].mean()

def classify_race(distance):
    if distance <= 5:
        return "speed"
    elif distance <= 15:
        return "threshold"
    else:
        return "endurance"

def predict_pace(distance_km, efficiency):
    
    # 1. Predicción base
    base_pace = model_eff.predict(pd.DataFrame({"efficiency": [efficiency]}))[0]
    
    # 2. Perfil de la carrera
    profile = classify_race(distance_km)
    
    # 3. Corrección por perfil (convertimos seg → min)
    bias_sec = bias_by_profile[profile]
    bias_min = bias_sec / 60.0
    
    # 4. Corrección final
    corrected_pace = base_pace - bias_min
    
    return corrected_pace

In [ ]:
predict_pace(3, 0.10)
predict_pace(10, 0.10)
predict_pace(21, 0.10)
predict_pace(42, 0.10)

In [ ]:
df_race_only["pace_pred_corr"] = df_race_only.apply(
    lambda row: predict_pace(row["distance_km"], row["efficiency"]),
    axis=1
)

df_race_only["error_corr_seg_km"] = (
    df_race_only["pace_pred_corr"] - df_race_only["pace_min_km"]
) * 60

df_race_only["abs_error_corr"] = df_race_only["error_corr_seg_km"].abs()

In [ ]:
df_race_only.groupby("race_profile").agg({
    "abs_error_corr": "mean",
    "error_corr_seg_km": "mean"
})

In [ ]:
df_race_only.sort_values("abs_error_corr", ascending=False)[[
    "date", "distance_km", "race_profile",
    "pace_min_km", "pace_pred_corr",
    "error_corr_seg_km"
]].head(10)

In [ ]:
df_race_only.columns

In [ ]:
df_race_only.sort_values("abs_error_corr", ascending=False)[[
    "session_date", "distance_km", "race_profile",
    "pace_min_km", "pace_pred_corr",
    "error_corr_seg_km"
]].head(10)

In [ ]:
df_race_only["is_outlier"] = df_race_only["abs_error_corr"] > 8

In [ ]:
df_race_only[df_race_only["is_outlier"]][[
    "session_date", "distance_km", "race_profile",
    "pace_min_km", "pace_pred_corr",
    "error_corr_seg_km"
]].sort_values("abs_error_corr", ascending=False)

In [ ]:
df_race_only[df_race_only["is_outlier"]][[
    "session_date", "distance_km", "race_profile",
    "pace_min_km", "pace_pred_corr",
    "error_corr_seg_km"
]].sort_values("abs_error_seg_km", ascending=False)

In [ ]:
df_race_only.columns.tolist()

In [ ]:
df_race_only[df_race_only["is_outlier"]][[
    "session_date", "distance_km", "race_profile",
    "pace_min_km", "pace_pred_corr",
    "error_corr_seg_km"
]].sort_values("error_corr_seg_km", ascending=False)

In [ ]:
df_race_final = df_race_only[~df_race_only["final_exclude"]].copy()

In [ ]:
exclude_dates = ["2025-11-30"]  # puedes ajustar luego

df_race_only["manual_exclude"] = df_race_only["session_date"].dt.strftime("%Y-%m-%d").isin(exclude_dates)

In [ ]:
df_race_only["final_exclude"] = df_race_only["is_outlier"] | df_race_only["manual_exclude"]

In [ ]:
df_race_final = df_race_only[~df_race_only["final_exclude"]].copy()

In [ ]:
df_race_final.shape

In [ ]:
from sklearn.linear_model import LinearRegression

X_final = df_race_final[["efficiency"]]
y_final = df_race_final["pace_min_km"]

model_eff_final = LinearRegression()
model_eff_final.fit(X_final, y_final)

In [ ]:
print(model_eff_final.coef_[0])
print(model_eff_final.intercept_)

In [ ]:
df_race_final["pace_pred_final"] = model_eff_final.predict(df_race_final[["efficiency"]])

df_race_final["error_final"] = (
    df_race_final["pace_pred_final"] - df_race_final["pace_min_km"]
) * 60

df_race_final["abs_error_final"] = df_race_final["error_final"].abs()

df_race_final["abs_error_final"].mean()

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_race_final,
    test_size=0.3,
    random_state=42
)

In [ ]:
print(len(df_train), len(df_test))

In [ ]:
from sklearn.linear_model import LinearRegression

X_train = df_train[["efficiency"]]
y_train = df_train["pace_min_km"]

model_test = LinearRegression()
model_test.fit(X_train, y_train)

In [ ]:
df_test["pace_pred"] = model_test.predict(df_test[["efficiency"]])

df_test["error_seg"] = (
    df_test["pace_pred"] - df_test["pace_min_km"]
) * 60

df_test["abs_error"] = df_test["error_seg"].abs()

df_test["abs_error"].mean()

In [ ]:
def runner_state(df_sessions):

    df = df_sessions.copy()
    
    # usar solo sesiones válidas
    df = df[df["valid_pace_hr"] == True]
    
    # calcular efficiency
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]
    
    # promedio simple (luego lo refinamos)
    eff = df["efficiency"].mean()
    
    return eff

In [ ]:
def predict_race(distance_km, df_sessions):
    
    eff = runner_state(df_sessions)
    
    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff]})
    )[0]
    
    return pace

In [ ]:
df_recent = df_model.sort_values("session_date").tail(20)

In [ ]:
predict_race(10, df_recent)

In [ ]:
df_model.columns.tolist()

In [ ]:
def runner_state(df_sessions):
    df = df_sessions.copy()

    # usar sesiones con datos válidos
    df = df.dropna(subset=["pace_min_km", "avg_hr"])

    # calcular efficiency
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]

    # estado actual = promedio simple
    eff = df["efficiency"].mean()

    return eff


def predict_race(distance_km, df_sessions):
    eff = runner_state(df_sessions)

    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff]})
    )[0]

    return pace

In [ ]:
df_recent = df_model.sort_values("session_date").tail(20)

predict_race(10, df_recent)

In [ ]:
def runner_state(df_sessions):

    df = df_sessions.copy()

    # datos válidos
    df = df.dropna(subset=["pace_min_km", "avg_hr"])

    # calcular efficiency
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]

    # quedarnos con las mejores sesiones (top 30%)
    df = df.sort_values("efficiency", ascending=False)
    df = df.head(int(len(df) * 0.3))

    eff = df["efficiency"].mean()

    return eff

In [ ]:
predict_race(10, df_recent)

In [ ]:
def runner_state(df_sessions):

    df = df_sessions.copy()

    # datos válidos
    df = df.dropna(subset=["pace_min_km", "avg_hr"])

    # SOLO sesiones relevantes (ritmos exigentes)
    df = df[df["pace_min_km"] < 4.3]

    # calcular efficiency
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]

    # top 50% de esas sesiones
    df = df.sort_values("efficiency", ascending=False)
    df = df.head(max(1, int(len(df) * 0.5)))

    eff = df["efficiency"].mean()

    return eff

In [ ]:
predict_race(10, df_recent)

In [ ]:
def runner_state(df_sessions):

    df = df_sessions.copy()

    df = df.dropna(subset=["pace_min_km", "avg_hr"])

    # calcular efficiency
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]

    # intento 1: sesiones rápidas
    df_fast = df[df["pace_min_km"] < 4.3]

    if len(df_fast) >= 3:
        df_use = df_fast
    else:
        # fallback: top 40% por eficiencia
        df = df.sort_values("efficiency", ascending=False)
        df_use = df.head(max(3, int(len(df) * 0.4)))

    eff = df_use["efficiency"].mean()

    return eff

In [ ]:
predict_race(10, df_recent)

In [ ]:
df_race_final["efficiency"].mean()

In [ ]:
def predict_race_v2(distance_km, df_sessions):

    eff_base = df_race_final["efficiency"].mean()
    eff_current = runner_state(df_sessions)

    # mezcla: 70% base + 30% estado actual
    eff = 0.7 * eff_base + 0.3 * eff_current

    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff]})
    )[0]

    return pace

In [ ]:
predict_race_v2(10, df_recent)

In [ ]:
def readiness_weight(df_sessions):
    df = df_sessions.copy()
    df = df.dropna(subset=["pace_min_km", "avg_hr"])

    # efficiency por sesión
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]

    # referencia (fitness base)
    eff_base = df_race_final["efficiency"].mean()

    # top 30% reciente (mejores señales)
    df = df.sort_values("efficiency", ascending=False)
    df_top = df.head(max(3, int(len(df) * 0.3)))

    eff_current = df_top["efficiency"].mean()

    # ratio vs base
    ratio = eff_current / eff_base

    # mapear ratio → peso [0.2, 0.6]
    if ratio >= 1.02:
        w = 0.6   # en forma (taper)
    elif ratio >= 0.98:
        w = 0.4   # normal
    else:
        w = 0.2   # fatiga

    return w, eff_current

In [ ]:
def predict_race_v3(distance_km, df_sessions):
    
    eff_base = df_race_final["efficiency"].mean()
    w, eff_current = readiness_weight(df_sessions)

    # mezcla dinámica
    eff = (1 - w) * eff_base + w * eff_current

    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff]})
    )[0]

    return pace, w, eff_base, eff_current

In [ ]:
predict_race_v3(10, df_recent)

In [ ]:
df_recent = df_model.sort_values("session_date").tail(30)

w, eff_current = readiness_weight(df_recent)

eff_base = df_race_final["efficiency"].mean()

ratio = eff_current / eff_base

ratio

In [ ]:
def interpret_state(ratio):
    if ratio >= 1.02:
        return "🔥 Pico de forma (listo para competir fuerte)"
    elif ratio >= 0.98:
        return "🟢 Buen estado (puedes rendir bien)"
    elif ratio >= 0.94:
        return "🟡 Estado medio (necesitas afinar)"
    else:
        return "🔴 Fatiga o carga alta (no estás en pico)"

interpret_state(ratio)

In [ ]:
def predict_race_date(distance_km, df_sessions, weeks_to_race):

    eff_base = df_race_final["efficiency"].mean()
    w, eff_current = readiness_weight(df_sessions)

    # mejora progresiva hacia tu base
    improvement = (eff_base - eff_current)

    # factor de mejora según semanas (0 → 1)
    k = min(1.0, weeks_to_race / 8)  # en 8 semanas puedes acercarte mucho

    eff_future = eff_current + k * improvement

    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff_future]})
    )[0]

    return pace, eff_current, eff_future

In [ ]:
predict_race_date(10, df_recent, 4)

In [ ]:
def predict_race_date(distance_km, df_sessions, weeks_to_race, taper=False):

    eff_base = df_race_final["efficiency"].mean()
    w, eff_current = readiness_weight(df_sessions)

    improvement = (eff_base - eff_current)

    k = min(1.0, weeks_to_race / 8)

    eff_future = eff_current + k * improvement

    # 🚀 si hay taper, empujamos hacia el pico
    if taper:
        eff_future = eff_future + 0.5 * (eff_base - eff_future)

    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff_future]})
    )[0]

    return pace, eff_current, eff_future

In [ ]:
predict_race_date(10, df_recent, 4, taper=True)

In [ ]:
if taper:
    eff_future = eff_future + 0.8 * (eff_base - eff_future)

In [ ]:
def predict_race_date(distance_km, df_sessions, weeks_to_race, taper=False):

    eff_base = df_race_final["efficiency"].mean()
    w, eff_current = readiness_weight(df_sessions)

    improvement = (eff_base - eff_current)

    k = min(1.0, weeks_to_race / 8)

    eff_future = eff_current + k * improvement

    # 🔥 ajuste fuerte de taper
    if taper:
        eff_future = eff_future + 0.8 * (eff_base - eff_future)

    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff_future]})
    )[0]

    return pace, eff_current, eff_future

In [ ]:
predict_race_date(10, df_recent, 4, taper=True)

In [ ]:
def tag_quality_sessions(df_sessions):
    df = df_sessions.copy()
    df = df.dropna(subset=["pace_min_km", "avg_hr"])

    # efficiency
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]

    # referencia: ritmo 10K estimado desde tu base
    eff_base = df_race_final["efficiency"].mean()
    pace_10k_est = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff_base]})
    )[0]

    # criterio de calidad:
    # sesiones cerca de ritmo competitivo (no rodajes suaves)
    df["is_quality"] = df["pace_min_km"] <= (pace_10k_est + 0.3)

    return df

In [ ]:
def runner_state_pro(df_sessions):

    df = tag_quality_sessions(df_sessions)

    # usar SOLO sesiones de calidad si hay suficientes
    df_q = df[df["is_quality"]]

    if len(df_q) >= 3:
        df_use = df_q
    else:
        # fallback: top 30% efficiency
        df = df.sort_values("efficiency", ascending=False)
        df_use = df.head(max(3, int(len(df) * 0.3)))

    eff = df_use["efficiency"].mean()

    return eff, len(df_q)

In [ ]:
def predict_race_v4(distance_km, df_sessions):

    eff_base = df_race_final["efficiency"].mean()
    eff_current, n_quality = runner_state_pro(df_sessions)

    # peso dinámico según calidad detectada
    if n_quality >= 5:
        w = 0.6
    elif n_quality >= 3:
        w = 0.4
    else:
        w = 0.2

    eff = (1 - w) * eff_base + w * eff_current

    pace = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff]})
    )[0]

    return pace, eff_current, n_quality, w

In [ ]:
predict_race_v4(10, df_recent)

In [ ]:
def training_gaps(df_sessions):

    df = tag_quality_sessions(df_sessions)

    n_quality = df["is_quality"].sum()

    # distribución de ritmos
    avg_pace = df["pace_min_km"].mean()

    # referencia
    eff_base = df_race_final["efficiency"].mean()
    pace_10k_est = model_eff_final.predict(
        pd.DataFrame({"efficiency": [eff_base]})
    )[0]

    gaps = []

    # GAP 1 — falta de sesiones de calidad
    if n_quality == 0:
        gaps.append("❌ No hay sesiones de calidad (tempo/intervalos)")

    elif n_quality < 3:
        gaps.append("⚠️ Pocas sesiones de calidad recientes")

    # GAP 2 — ritmos muy lejos de competencia
    if avg_pace > pace_10k_est + 0.8:
        gaps.append("🐢 Ritmos generales demasiado lentos")

    return gaps, n_quality, pace_10k_est

In [ ]:
def training_recommendations(df_sessions):

    gaps, n_quality, pace_10k_est = training_gaps(df_sessions)

    recs = []

    for g in gaps:

        if "No hay sesiones" in g:
            recs.append(
                f"➡️ Introducir 2 sesiones/semana tipo tempo o intervalos cerca de {pace_10k_est:.2f} min/km"
            )

        if "Pocas sesiones" in g:
            recs.append(
                f"➡️ Aumentar a 3 sesiones de calidad por semana"
            )

        if "Ritmos generales" in g:
            recs.append(
                f"➡️ Incluir rodajes progresivos acercándose a {pace_10k_est:.2f} min/km"
            )

    if len(recs) == 0:
        recs.append("✅ Entrenamiento bien balanceado")

    return gaps, recs

In [ ]:
gaps, recs = training_recommendations(df_recent)

gaps, recs

In [ ]:
def weekly_snapshot(df_sessions):

    eff_base = df_race_final["efficiency"].mean()
    w, eff_current = readiness_weight(df_sessions)
    ratio = eff_current / eff_base

    pace, _, _, _ = predict_race_v4(10, df_sessions)

    gaps, recs = training_recommendations(df_sessions)

    snapshot = {
        "eff_base": eff_base,
        "eff_current": eff_current,
        "ratio": ratio,
        "pred_10k_pace": pace,
        "gaps": gaps,
        "recs": recs
    }

    return snapshot

In [ ]:
snapshot = weekly_snapshot(df_recent)
snapshot

In [ ]:
df_model.to_csv("df_model.csv", index=False)
df_race_final.to_csv("df_race_final.csv", index=False)

In [ ]:
df_model.to_csv("df_model.csv", index=False)
df_race_final.to_csv("df_race_final.csv", index=False)

In [ ]:
import pandas as pd
print(pd)
print(pd.__file__)
print(pd.__version__)


In [ ]:
df_sessions

In [ ]:
print("ok")

In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
df_sessions

In [ ]:
data = [
    {"session_date": "2025-10-01", "distance_km": 10.3, "duration_min": 38.0, "pace_min_km": 3.68, "avg_hr": 151},
    {"session_date": "2025-10-05", "distance_km": 21.0, "duration_min": 95.0, "pace_min_km": 4.52, "avg_hr": 148},
    {"session_date": "2025-10-12", "distance_km": 28.0, "duration_min": 130.0, "pace_min_km": 4.64, "avg_hr": 145},
]

import pandas as pd

df_sessions = pd.DataFrame(data)
df_sessions

In [ ]:
df_sessions["efficiency"] = (60 / df_sessions["pace_min_km"]) / df_sessions["avg_hr"]
df_sessions

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_sessions[["efficiency"]]
y = df_sessions["pace_min_km"]

model = LinearRegression()
model.fit(X, y)

model.coef_, model.intercept_

In [ ]:
def predict_pace(eff):
    return model.predict([[eff]])[0]

predict_pace(0.09)

In [ ]:
def predict_pace(eff):
    import pandas as pd
    return model.predict(pd.DataFrame({"efficiency": [eff]}))[0]

In [ ]:
eff_base = df_sessions["efficiency"].mean()
eff_current = df_sessions.tail(2)["efficiency"].mean()

ratio = eff_current / eff_base

eff_base, eff_current, ratio

In [ ]:
eff_used = 0.7 * eff_base + 0.3 * eff_current

pace_pred = predict_pace(eff_used)

eff_used, pace_pred

In [ ]:
def interpret_state(ratio):
    if ratio >= 1.02:
        return "🔥 Pico de forma"
    elif ratio >= 0.98:
        return "🟢 Buen estado"
    elif ratio >= 0.94:
        return "🟡 Estado medio"
    else:
        return "🔴 Fatiga / carga alta"

interpret_state(ratio)

In [ ]:
import os

for root, dirs, files in os.walk("."):
    print(root)
    break

In [ ]:
os.listdir()

In [ ]:
os.listdir("..")

In [ ]:
os.listdir("../data")

In [ ]:
os.listdir("../data/raw")

In [ ]:
os.listdir("../data/raw/polar")

In [ ]:
files = os.listdir("../data/raw/polar")
files[-10:]

In [ ]:
import json
import os

folder = "../data/raw/polar"

sessions = []

files = os.listdir(folder)

for f in files[:200]:  # empezamos con 200 para no colgar nada
    path = os.path.join(folder, f)
    try:
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)
            sessions.append(data)
    except:
        continue

len(sessions)

In [ ]:
import pandas as pd

rows = []

for s in sessions:
    try:
        row = {
            "session_date": s.get("startTime", "")[:10],
            "duration_s": s.get("duration", 0),
            "distance_km": s.get("distance", 0) / 1000,
            "avg_hr": s.get("heartRate", {}).get("average", None),
            "max_hr": s.get("heartRate", {}).get("max", None),
            "calories": s.get("calories", None),
            "name": s.get("name", "")
        }
        rows.append(row)
    except:
        continue

df_real = pd.DataFrame(rows)

df_real.head()

In [ ]:
def parse_duration(x):
    try:
        return float(x.replace("PT", "").replace("S", ""))
    except:
        return None

df_real["duration_s"] = df_real["duration_s"].apply(parse_duration)

df_real.head()

In [ ]:
df_real["duration_min"] = df_real["duration_s"] / 60

df_real.head()

In [ ]:
df_real["pace_min_km"] = df_real["duration_min"] / df_real["distance_km"]

df_real[["distance_km", "duration_min", "pace_min_km"]].head()

In [ ]:
import json, os

folder = "../data/raw/polar"
sessions = []

files = os.listdir(folder)

for f in files[:200]:
    path = os.path.join(folder, f)
    try:
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)
            sessions.append(data)
    except:
        continue

In [ ]:
import pandas as pd

rows = []

for s in sessions:
    try:
        row = {
            "session_date": s.get("startTime", "")[:10],
            "duration_s": s.get("duration", 0),
            "distance_km": s.get("distance", 0) / 1000,
            "avg_hr": s.get("heartRate", {}).get("average", None),
        }
        rows.append(row)
    except:
        continue

df_real = pd.DataFrame(rows)

In [ ]:
def parse_duration(x):
    try:
        return float(x.replace("PT", "").replace("S", ""))
    except:
        return None

df_real["duration_s"] = df_real["duration_s"].apply(parse_duration)
df_real["duration_min"] = df_real["duration_s"] / 60


In [ ]:
df_real["pace_min_km"] = df_real["duration_min"] / df_real["distance_km"]

In [ ]:
df_real["pace_min_km"] = df_real["duration_min"] / df_real["distance_km"]

df_real[["distance_km", "duration_min", "pace_min_km"]].head()

In [ ]:
df_real["efficiency"] = (60 / df_real["pace_min_km"]) / df_real["avg_hr"]

df_real[["pace_min_km", "avg_hr", "efficiency"]].head()

In [ ]:
import json

sample_file = os.path.join(folder, files[0])

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

sample.keys()

In [ ]:
rows = []

for s in sessions:
    try:
        row = {
            "session_date": s.get("startTime", "")[:10],
            "duration_s": s.get("duration", 0),
            "distance_km": s.get("distance", 0) / 1000,
            "avg_hr": s.get("averageHeartRate", None),
            "max_hr": s.get("maximumHeartRate", None),
        }
        rows.append(row)
    except:
        continue

df_real = pd.DataFrame(rows)

# limpiar duración otra vez
def parse_duration(x):
    try:
        return float(x.replace("PT", "").replace("S", ""))
    except:
        return None

df_real["duration_s"] = df_real["duration_s"].apply(parse_duration)
df_real["duration_min"] = df_real["duration_s"] / 60

# pace
df_real["pace_min_km"] = df_real["duration_min"] / df_real["distance_km"]

# efficiency
df_real["efficiency"] = (60 / df_real["pace_min_km"]) / df_real["avg_hr"]

df_real[["pace_min_km", "avg_hr", "efficiency"]].head()

In [ ]:
eff_base = df_real["efficiency"].mean()

eff_recent = df_real.tail(20)["efficiency"].mean()

ratio = eff_recent / eff_base

eff_base, eff_recent, ratio

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_real[["efficiency"]]
y = df_real["pace_min_km"]

model = LinearRegression()
model.fit(X, y)

def predict_pace(eff):
    import pandas as pd
    return model.predict(pd.DataFrame({"efficiency": [eff]}))[0]

# usar tu estado actual
pace_pred = predict_pace(eff_recent)

pace_pred


In [ ]:
df_clean = df_real.dropna(subset=["efficiency", "pace_min_km"])

len(df_real), len(df_clean)

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_clean[["efficiency"]]
y = df_clean["pace_min_km"]

model = LinearRegression()
model.fit(X, y)

def predict_pace(eff):
    import pandas as pd
    return model.predict(pd.DataFrame({"efficiency": [eff]}))[0]

In [ ]:
import numpy as np

df_clean = df_real.copy()

# convertir a numérico por si algo quedó raro
df_clean["efficiency"] = pd.to_numeric(df_clean["efficiency"], errors="coerce")
df_clean["pace_min_km"] = pd.to_numeric(df_clean["pace_min_km"], errors="coerce")

# eliminar NaN e infinitos
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=["efficiency", "pace_min_km"])

len(df_clean)

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_clean[["efficiency"]]
y = df_clean["pace_min_km"]

model = LinearRegression()
model.fit(X, y)

In [ ]:
def predict_pace(eff):
    import pandas as pd
    return model.predict(pd.DataFrame({"efficiency": [eff]}))[0]

pace_pred = predict_pace(eff_recent)

pace_pred

In [ ]:
df_perf = df_clean[
    (df_clean["distance_km"] >= 8) &
    (df_clean["pace_min_km"] <= 5.0) &
    (df_clean["avg_hr"] >= 140)
]

len(df_clean), len(df_perf)

In [ ]:
import numpy as np
import pandas as pd

df_clean = df_real.copy()

df_clean["efficiency"] = pd.to_numeric(df_clean["efficiency"], errors="coerce")
df_clean["pace_min_km"] = pd.to_numeric(df_clean["pace_min_km"], errors="coerce")
df_clean["distance_km"] = pd.to_numeric(df_clean["distance_km"], errors="coerce")
df_clean["avg_hr"] = pd.to_numeric(df_clean["avg_hr"], errors="coerce")

df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=["efficiency", "pace_min_km", "distance_km", "avg_hr"])

df_clean.shape

In [ ]:
import json, os
import pandas as pd
import numpy as np

# 1. Cargar JSON
folder = "../data/raw/polar"
files = os.listdir(folder)

sessions = []
for f in files[:200]:
    try:
        with open(os.path.join(folder, f), "r", encoding="utf-8") as file:
            sessions.append(json.load(file))
    except:
        continue

# 2. Crear DataFrame
rows = []
for s in sessions:
    try:
        rows.append({
            "session_date": s.get("startTime", "")[:10],
            "duration_s": s.get("duration", 0),
            "distance_km": s.get("distance", 0) / 1000,
            "avg_hr": s.get("averageHeartRate", None),
        })
    except:
        continue

df_real = pd.DataFrame(rows)

# 3. Limpiar duración
def parse_duration(x):
    try:
        return float(x.replace("PT", "").replace("S", ""))
    except:
        return None

df_real["duration_s"] = df_real["duration_s"].apply(parse_duration)
df_real["duration_min"] = df_real["duration_s"] / 60

# 4. Crear pace y efficiency
df_real["pace_min_km"] = df_real["duration_min"] / df_real["distance_km"]
df_real["efficiency"] = (60 / df_real["pace_min_km"]) / df_real["avg_hr"]

# 5. Limpiar datos
df_clean = df_real.copy()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=["efficiency", "pace_min_km", "distance_km", "avg_hr"])

df_clean.shape

In [ ]:
df_perf = df_clean[
    (df_clean["distance_km"] >= 8) &
    (df_clean["pace_min_km"] <= 5.0) &
    (df_clean["avg_hr"] >= 140)
]

In [ ]:
df_clean.shape

In [ ]:
df_perf = df_clean[
    (df_clean["distance_km"] >= 8) &
    (df_clean["pace_min_km"] <= 5.0) &
    (df_clean["avg_hr"] >= 140)
]

len(df_clean), len(df_perf)

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_perf[["efficiency"]]
y = df_perf["pace_min_km"]

model_perf = LinearRegression()
model_perf.fit(X, y)

def predict_pace_perf(eff):
    import pandas as pd
    return model_perf.predict(pd.DataFrame({"efficiency": [eff]}))[0]

In [ ]:
pace_perf = predict_pace_perf(eff_recent)

pace_perf

In [ ]:
eff_base = df_clean["efficiency"].mean()
eff_recent = df_clean.tail(20)["efficiency"].mean()

eff_base, eff_recent

In [ ]:
pace_perf = predict_pace_perf(eff_recent)

pace_perf

In [ ]:
gap = pace_pred - pace_perf
gap

In [ ]:
pace_pred = predict_pace(eff_recent)

pace_pred

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_clean[["efficiency"]]
y = df_clean["pace_min_km"]

model = LinearRegression()
model.fit(X, y)

def predict_pace(eff):
    import pandas as pd
    return model.predict(pd.DataFrame({"efficiency": [eff]}))[0]

pace_pred = predict_pace(eff_recent)

In [ ]:
gap = pace_pred - pace_perf
gap * 60

In [ ]:
def build_pipeline(n_files=200):
    import json, os
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LinearRegression

    folder = "../data/raw/polar"
    files = os.listdir(folder)

    sessions = []
    for f in files[:n_files]:
        try:
            with open(os.path.join(folder, f), "r", encoding="utf-8") as file:
                sessions.append(json.load(file))
        except:
            continue

    rows = []
    for s in sessions:
        try:
            rows.append({
                "session_date": s.get("startTime", "")[:10],
                "duration_s": s.get("duration", 0),
                "distance_km": s.get("distance", 0) / 1000,
                "avg_hr": s.get("averageHeartRate", None),
            })
        except:
            continue

    df = pd.DataFrame(rows)

    def parse_duration(x):
        try:
            return float(x.replace("PT", "").replace("S", ""))
        except:
            return None

    df["duration_s"] = df["duration_s"].apply(parse_duration)
    df["duration_min"] = df["duration_s"] / 60
    df["pace_min_km"] = df["duration_min"] / df["distance_km"]
    df["efficiency"] = (60 / df["pace_min_km"]) / df["avg_hr"]

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["efficiency", "pace_min_km", "distance_km", "avg_hr"])

    df_perf = df[
        (df["distance_km"] >= 8) &
        (df["pace_min_km"] <= 5.0) &
        (df["avg_hr"] >= 140)
    ]

    model = LinearRegression()
    model.fit(df_perf[["efficiency"]], df_perf["pace_min_km"])

    eff_recent = df.tail(20)["efficiency"].mean()
    pace = model.predict(pd.DataFrame({"efficiency": [eff_recent]}))[0]

    return pace, df, df_perf

In [ ]:
pace, df_all, df_perf = build_pipeline()

pace

In [ ]:
def runner_state(df):
    eff_base = df["efficiency"].mean()
    eff_recent = df.tail(20)["efficiency"].mean()
    ratio = eff_recent / eff_base

    if ratio >= 1.02:
        state = "🔥 Pico de forma"
    elif ratio >= 0.98:
        state = "🟢 Buen estado"
    elif ratio >= 0.94:
        state = "🟡 Estado medio"
    else:
        state = "🔴 Fatiga"

    return eff_base, eff_recent, ratio, state

In [ ]:
eff_base, eff_recent, ratio, state = runner_state(df_all)

eff_base, eff_recent, ratio, state

In [ ]:
def runner_summary(df, pace_perf):
    eff_base = df["efficiency"].mean()
    eff_recent = df.tail(20)["efficiency"].mean()
    ratio = eff_recent / eff_base

    if ratio >= 1.02:
        state = "🔥 Pico de forma"
    elif ratio >= 0.98:
        state = "🟢 Buen estado"
    elif ratio >= 0.94:
        state = "🟡 Estado medio"
    else:
        state = "🔴 Fatiga"

    summary = {
        "eff_base": eff_base,
        "eff_recent": eff_recent,
        "ratio": ratio,
        "state": state,
        "predicted_pace": pace_perf
    }

    return summary

In [ ]:
summary = runner_summary(df_all, pace)

summary